In [ ]:
from pathlib import Path
CSV_FILE = Path('path')
OUT_DIR = Path('path')
if str(CSV_FILE) == 'path' or str(OUT_DIR) == 'path':
    raise ValueError('Set CSV_FILE to the input CSV and OUT_DIR to an output directory.')
if not CSV_FILE.is_file():
    raise FileNotFoundError(CSV_FILE)
import os, sys, subprocess
os.environ['USE_TF'] = '0'
os.environ['USE_FLAX'] = '0'
os.environ['TOKENIZERS_PARALLELISM'] = 'false'
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'transformers==4.57.1', 'datasets>=3,<5', 'accelerate>=1,<2', 'scikit-learn>=1.5,<1.8', 'pandas', 'numpy', 'torch', 'joblib'])
import json, time, re, hashlib, uuid
from datetime import datetime
import numpy as np
import pandas as pd
import torch
import joblib
from IPython.display import display
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.svm import LinearSVC
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import Pipeline as SKPipeline
from sklearn.calibration import CalibratedClassifierCV
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, matthews_corrcoef, roc_auc_score, confusion_matrix, classification_report
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AutoModelForMaskedLM, DataCollatorForLanguageModeling, Trainer, TrainingArguments, EarlyStoppingCallback, set_seed
from datasets import Dataset
set_seed(42)
DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
if DEVICE != 'cuda':
    raise RuntimeError('A CUDA GPU is required for this training notebook.')
RUN_DIR = OUT_DIR / ('run_' + datetime.now().strftime('%Y%m%d_%H%M%S') + '_' + uuid.uuid4().hex[:8])
TABLE_DIR = RUN_DIR / 'tables'
MODEL_DIR = RUN_DIR / 'models'
PRED_DIR = RUN_DIR / 'predictions'
for directory in [TABLE_DIR, MODEL_DIR, PRED_DIR]:
    directory.mkdir(parents=True, exist_ok=False)
WALL_TIMES = {}
MODEL_ORDER = ['TF-IDF+SVM', 'TF-IDF+LR', 'BERT-base', 'ModernBERT', 'ModernBERT+TAPT', 'ModernBERT+AdaptFT', 'ModernBERT+Fusion']

def compute_hf_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return {'accuracy': accuracy_score(labels, predictions), 'macro_f1': f1_score(labels, predictions, labels=[0, 1], average='macro', zero_division=0)}

def compute_metrics(y_true, y_pred, y_prob=None):
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {'Accuracy': accuracy_score(y_true, y_pred), 'Macro-F1': f1_score(y_true, y_pred, average='macro', zero_division=0), 'Precision': precision_score(y_true, y_pred, average='macro', zero_division=0), 'Recall': recall_score(y_true, y_pred, average='macro', zero_division=0), 'Specificity': tn / (tn + fp + 1e-09), 'MCC': matthews_corrcoef(y_true, y_pred), 'ROC-AUC': roc_auc_score(y_true, y_prob) if y_prob is not None else None}

def hf_dataset(df_):
    return Dataset.from_dict({'text': df_.text.tolist(), 'label': df_.label.tolist()})

def train_hf_model(model_name, train_ds, val_ds, test_ds, tokenizer=None, max_len=512, epochs=3, batch=32, lr=2e-05, save_path=None, freeze_patterns=None, seed=42, skip_if_exists=False):
    set_seed(seed)
    t_start = time.time()
    local = Path(str(model_name)).exists()
    tok = tokenizer or AutoTokenizer.from_pretrained(model_name, local_files_only=local)

    def tok_fn(batch):
        return tok(batch['text'], truncation=True, padding='max_length', max_length=max_len)
    tok_tr = train_ds.map(tok_fn, batched=True, batch_size=512, remove_columns=['text'])
    tok_va = val_ds.map(tok_fn, batched=True, batch_size=512, remove_columns=['text'])
    tok_te = test_ds.map(tok_fn, batched=True, batch_size=512, remove_columns=['text'])
    model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=2, ignore_mismatched_sizes=True, local_files_only=local)
    if freeze_patterns:
        frozen = trained = 0
        for name, param in model.named_parameters():
            sf = any((fp in name for fp in freeze_patterns))
            param.requires_grad = not sf
            if sf:
                frozen += param.numel()
            else:
                trained += param.numel()
        total = frozen + trained
    if hasattr(model.config, 'reference_compile'):
        model.config.reference_compile = False
    model.to(DEVICE)
    out_tag = str(model_name).replace('/', '_').split('\\')[-1].split('/')[-1]
    args = TrainingArguments(output_dir=str(MODEL_DIR / out_tag), num_train_epochs=epochs, per_device_train_batch_size=8, gradient_accumulation_steps=4, per_device_eval_batch_size=16, learning_rate=lr, warmup_ratio=0.1, weight_decay=0.01, eval_strategy='epoch', save_strategy='epoch', save_total_limit=1, load_best_model_at_end=True, metric_for_best_model='macro_f1', greater_is_better=True, fp16=DEVICE == 'cuda', dataloader_num_workers=0, report_to='none', logging_steps=100, seed=seed, data_seed=seed)
    trainer = Trainer(model=model, args=args, train_dataset=tok_tr, eval_dataset=tok_va, compute_metrics=compute_hf_metrics, callbacks=[EarlyStoppingCallback(early_stopping_patience=2)])
    trainer.train()
    out = trainer.predict(tok_te)
    preds = np.argmax(out.predictions, axis=-1)
    probs = torch.softmax(torch.tensor(out.predictions.astype(np.float32)), dim=-1).numpy()[:, 1]
    elapsed = round(time.time() - t_start, 1)
    if save_path:
        trainer.save_model(str(save_path))
        tok.save_pretrained(str(save_path))
    del model
    torch.cuda.empty_cache()
    return (tok, preds, probs, elapsed)

def extract_all_embeddings(texts, tokenizer, model, batch_size=8, max_len=512):
    cls_list = []
    mean_list = []
    cm_list = []
    l2_list = []
    for i in range(0, len(texts), batch_size):
        enc = tokenizer(texts[i:i + batch_size], return_tensors='pt', truncation=True, padding='max_length', max_length=max_len)
        enc_gpu = {k: v.to(DEVICE) for k, v in enc.items()}
        with torch.no_grad():
            out = model(**enc_gpu, output_hidden_states=True, return_dict=True)
        last = out.hidden_states[-1]
        prev = out.hidden_states[-2]
        mask = enc['attention_mask'].unsqueeze(-1).float().to(DEVICE)
        cls_e = last[:, 0, :]
        mean_e = (last * mask).sum(1) / mask.sum(1).clamp(min=1e-09)
        cm_e = torch.cat([cls_e, mean_e], dim=-1)
        l2_e = (last[:, 0, :] + prev[:, 0, :]) / 2
        cls_list.append(cls_e.cpu().float().numpy())
        mean_list.append(mean_e.cpu().float().numpy())
        cm_list.append(cm_e.cpu().float().numpy())
        l2_list.append(l2_e.cpu().float().numpy())
    return {'cls': np.vstack(cls_list), 'mean': np.vstack(mean_list), 'cls_mean': np.vstack(cm_list), 'last2': np.vstack(l2_list)}

def meta_features(df_, thumbs_available=False):
    feats = [np.log1p(df_['review_length'].values.astype(np.float32)), np.sqrt(df_['review_length'].values.astype(np.float32)), (df_['review_length'].values > 100).astype(np.float32), (df_['review_length'].values < 20).astype(np.float32), df_['has_dev_response'].values.astype(np.float32)]
    if thumbs_available:
        feats.insert(0, np.log1p(df_['thumbs_up'].fillna(0).clip(lower=0).values.astype(np.float32)))
    return np.stack(feats, axis=1)
raw = pd.read_csv(CSV_FILE)
raw['row_id'] = np.arange(len(raw))
TEXT_COL = 'Review_text'
RATING_COL = 'Rating'
raw = raw.dropna(subset=[TEXT_COL, RATING_COL])
raw[TEXT_COL] = raw[TEXT_COL].astype(str).str.strip()
raw[RATING_COL] = pd.to_numeric(raw[RATING_COL], errors='coerce')
raw = raw.dropna(subset=[RATING_COL])
raw = raw[raw[RATING_COL].isin([1, 2, 3, 4, 5])].copy()
raw[RATING_COL] = raw[RATING_COL].astype(int)
raw = raw[raw[RATING_COL] != 3].copy()
raw = raw[raw[TEXT_COL].str.split().str.len() >= 5].copy()
raw['label'] = (raw[RATING_COL] >= 4).astype(int)
raw['text'] = raw[TEXT_COL]
raw['review_length'] = raw['text'].str.split().str.len()
THUMBS_COL = next((c for c in raw.columns if 'thumb' in c.lower() and 'up' in c.lower()), None)
if THUMBS_COL:
    raw['thumbs_up'] = pd.to_numeric(raw[THUMBS_COL], errors='coerce').fillna(0).clip(lower=0, upper=500)
    THUMBS_AVAILABLE = True
else:
    raw['thumbs_up'] = np.nan
    THUMBS_AVAILABLE = False
DEV_COL = next((c for c in raw.columns if 'developer' in c.lower() or 'dev_reply' in c.lower()), None)
raw['has_dev_response'] = raw[DEV_COL].notna().astype(int) if DEV_COL else 0
n_before = len(raw)
raw = raw.drop_duplicates(subset=['text']).copy()
if len(raw) < n_before:
    pass
train_df, tmp = train_test_split(raw, test_size=0.3, random_state=42, stratify=raw['label'])
val_df, test_df = train_test_split(tmp, test_size=0.5, random_state=42, stratify=tmp['label'])
for df_, name, n in [(train_df, 'train', 25000), (val_df, 'val', 5000), (test_df, 'test', 5000)]:
    if len(df_) > n:
        df_ = df_.sample(n, random_state=42).reset_index(drop=True)
        if name == 'train':
            train_df = df_
        elif name == 'val':
            val_df = df_
        else:
            test_df = df_
assert len(set(train_df.text) & set(test_df.text)) == 0, 'HATA: train∩test örtüşüyor!'
assert len(set(val_df.text) & set(test_df.text)) == 0, 'HATA: val∩test örtüşüyor!'
stat_df = pd.DataFrame({'Split': ['Train', 'Val', 'Test', 'Total'], 'N': [len(train_df), len(val_df), len(test_df), len(raw)], 'Positive': [int(train_df.label.sum()), int(val_df.label.sum()), int(test_df.label.sum()), int(raw.label.sum())], 'Pos%': [f'{train_df.label.mean() * 100:.1f}', f'{val_df.label.mean() * 100:.1f}', f'{test_df.label.mean() * 100:.1f}', f'{raw.label.mean() * 100:.1f}']})
stat_df.to_csv(TABLE_DIR / 'table_dataset_stats.csv', index=False)
display(stat_df)
results = {}
tr_ds = hf_dataset(train_df)
va_ds = hf_dataset(val_df)
te_ds = hf_dataset(test_df)
for split, frame in [('train', train_df), ('validation', val_df), ('test', test_df)]:
    frame['text_hash'] = frame['text'].map(lambda value: hashlib.sha256(value.encode('utf-8')).hexdigest())
    frame[['row_id', 'text_hash', 'label']].to_csv(TABLE_DIR / (split + '_manifest.csv'), index=False)
assert not set(train_df.text_hash) & set(val_df.text_hash)
for mname, clf in [('TF-IDF+SVM', CalibratedClassifierCV(LinearSVC(C=1.0, max_iter=2000))), ('TF-IDF+LR', LogisticRegression(C=1.0, max_iter=500, n_jobs=-1))]:
    pipe = SKPipeline([('tfidf', TfidfVectorizer(max_features=50000, ngram_range=(1, 2), sublinear_tf=True)), ('clf', clf)])
    t0 = time.time()
    pipe.fit(train_df.text, train_df.label)
    pred = pipe.predict(test_df.text)
    prob = pipe.predict_proba(test_df.text)[:, 1]
    t = round(time.time() - t0, 1)
    results[mname] = compute_metrics(test_df.label, pred, prob)
    results[mname]['TrainTime(s)'] = t
    WALL_TIMES[mname] = t
    if mname == 'TF-IDF+SVM':
        svm_pipe, svm_pred, svm_prob = (pipe, pred, prob)
    else:
        lr_pipe, lr_pred, lr_prob = (pipe, pred, prob)
BERT_PATH = MODEL_DIR / 'bert_finetuned'
bert_tok, bert_pred, bert_prob, bert_time = train_hf_model('bert-base-uncased', tr_ds, va_ds, te_ds, max_len=512, epochs=3, batch=32, seed=42, save_path=BERT_PATH, skip_if_exists=False)
results['BERT-base'] = compute_metrics(test_df.label, bert_pred, bert_prob)
results['BERT-base']['TrainTime(s)'] = bert_time
WALL_TIMES['BERT-base'] = bert_time
MODERNBERT = 'answerdotai/ModernBERT-base'
MBERT_PATH = MODEL_DIR / 'modernbert_base'
mbert_tok, mbert_pred, mbert_prob, mbert_time = train_hf_model(MODERNBERT, tr_ds, va_ds, te_ds, max_len=512, epochs=3, batch=32, seed=42, save_path=MBERT_PATH, skip_if_exists=False)
results['ModernBERT'] = compute_metrics(test_df.label, mbert_pred, mbert_prob)
results['ModernBERT']['TrainTime(s)'] = mbert_time
WALL_TIMES['ModernBERT'] = mbert_time
TAPT_CKPT = MODEL_DIR / 'tapt_ckpt'
if TAPT_CKPT.exists():
    WALL_TIMES['TAPT_pretrain'] = 0.0
    tapt_tok = AutoTokenizer.from_pretrained(str(TAPT_CKPT), local_files_only=True)
else:
    t0 = time.time()
    tapt_tok = AutoTokenizer.from_pretrained(MODERNBERT)
    tapt_raw = Dataset.from_dict({'text': train_df.text.tolist()[:5000]})

    def _tok_mlm(batch):
        return tapt_tok(batch['text'], truncation=True, padding='max_length', max_length=512)
    tapt_ds = tapt_raw.map(_tok_mlm, batched=True, batch_size=512, remove_columns=['text'])
    mlm = AutoModelForMaskedLM.from_pretrained(MODERNBERT)
    if hasattr(mlm.config, 'reference_compile'):
        mlm.config.reference_compile = False
    mlm.to(DEVICE)
    coll = DataCollatorForLanguageModeling(tokenizer=tapt_tok, mlm=True, mlm_probability=0.15)
    Trainer(model=mlm, args=TrainingArguments(output_dir=str(MODEL_DIR / 'tapt_tmp'), num_train_epochs=1, per_device_train_batch_size=8, gradient_accumulation_steps=4, learning_rate=0.0001, warmup_ratio=0.1, fp16=DEVICE == 'cuda', logging_steps=100, save_steps=9999, report_to='none', dataloader_num_workers=0, seed=42, data_seed=42), train_dataset=tapt_ds, data_collator=coll).train()
    mlm.save_pretrained(str(TAPT_CKPT))
    tapt_tok.save_pretrained(str(TAPT_CKPT))
    WALL_TIMES['TAPT_pretrain'] = round(time.time() - t0, 1)
    del mlm
    torch.cuda.empty_cache()
TAPT_FT_PATH = MODEL_DIR / 'tapt_finetuned'
t_ft = time.time()
_, tapt_pred, tapt_prob, tapt_ft_time = train_hf_model(str(TAPT_CKPT), tr_ds, va_ds, te_ds, tokenizer=tapt_tok, max_len=512, epochs=3, batch=32, seed=42, save_path=TAPT_FT_PATH, skip_if_exists=False)
results['ModernBERT+TAPT'] = compute_metrics(test_df.label, tapt_pred, tapt_prob)
results['ModernBERT+TAPT']['TrainTime(s)'] = tapt_ft_time
WALL_TIMES['TAPT_ft'] = tapt_ft_time
ADAPTFT_PATH = MODEL_DIR / 'adaptft_finetuned'
_probe = AutoModelForSequenceClassification.from_pretrained(str(TAPT_CKPT), num_labels=2, ignore_mismatched_sizes=True, local_files_only=True)
layer_idx = set()
for name, _ in _probe.named_parameters():
    for part in name.split('.'):
        if part.isdigit():
            layer_idx.add(int(part))
            break
total_l = max(layer_idx) if layer_idx else 11
freeze_until = total_l - 3
del _probe
torch.cuda.empty_cache()
FREEZE_PAT = ['embeddings'] + [f'layers.{i}.' for i in range(freeze_until + 1)] + [f'layer.{i}.' for i in range(freeze_until + 1)] + [f'encoder.layer.{i}.' for i in range(freeze_until + 1)]
_, adaptft_pred, adaptft_prob, adaptft_time = train_hf_model(str(TAPT_CKPT), tr_ds, va_ds, te_ds, tokenizer=tapt_tok, max_len=512, epochs=5, batch=32, lr=2e-05, freeze_patterns=FREEZE_PAT, seed=42, save_path=ADAPTFT_PATH, skip_if_exists=False)
results['ModernBERT+AdaptFT'] = compute_metrics(test_df.label, adaptft_pred, adaptft_prob)
results['ModernBERT+AdaptFT']['TrainTime(s)'] = adaptft_time
WALL_TIMES['AdaptFT'] = adaptft_time
t_fusion_total = time.time()
emb_model = AutoModelForSequenceClassification.from_pretrained(str(TAPT_FT_PATH), num_labels=2, ignore_mismatched_sizes=True, local_files_only=True)
if hasattr(emb_model.config, 'reference_compile'):
    emb_model.config.reference_compile = False
emb_model.to(DEVICE)
emb_model.eval()
emb_tok = AutoTokenizer.from_pretrained(str(TAPT_FT_PATH), local_files_only=True)
t_emb = time.time()
tr_emb = extract_all_embeddings(train_df.text.tolist(), emb_tok, emb_model)
va_emb = extract_all_embeddings(val_df.text.tolist(), emb_tok, emb_model)
te_emb = extract_all_embeddings(test_df.text.tolist(), emb_tok, emb_model)
del emb_model
torch.cuda.empty_cache()
emb_time = round(time.time() - t_emb, 1)
best_val_f1 = -1.0
best_cfg = None
STRATEGIES = ['cls', 'mean', 'cls_mean', 'last2']
for strategy in STRATEGIES:
    X_tr = np.hstack([tr_emb[strategy], meta_features(train_df, THUMBS_AVAILABLE)])
    X_va = np.hstack([va_emb[strategy], meta_features(val_df, THUMBS_AVAILABLE)])
    for C in [1.0, 5.0, 10.0]:
        pipe = SKPipeline([('sc', StandardScaler()), ('clf', LogisticRegression(C=C, max_iter=1000, random_state=42, n_jobs=-1))])
        pipe.fit(X_tr, train_df.label)
        vf1 = f1_score(val_df.label, pipe.predict(X_va), average='macro')
        if vf1 > best_val_f1:
            best_val_f1 = vf1
            best_cfg = {'strategy': strategy, 'C': C, 'pipe': pipe}
best_s = best_cfg['strategy']
X_tr_f = np.hstack([tr_emb[best_s], meta_features(train_df, THUMBS_AVAILABLE)])
X_te_f = np.hstack([te_emb[best_s], meta_features(test_df, THUMBS_AVAILABLE)])
final_pipe = SKPipeline([('sc', StandardScaler()), ('clf', LogisticRegression(C=best_cfg['C'], max_iter=1000, random_state=42, n_jobs=-1))])
final_pipe.fit(X_tr_f, train_df.label)
best_fusion_pred = final_pipe.predict(X_te_f)
best_fusion_prob = final_pipe.predict_proba(X_te_f)[:, 1]
results['ModernBERT+Fusion'] = compute_metrics(test_df.label, best_fusion_pred, best_fusion_prob)
all_preds = {'TF-IDF+SVM': svm_pred, 'TF-IDF+LR': lr_pred, 'BERT-base': bert_pred, 'ModernBERT': mbert_pred, 'ModernBERT+TAPT': tapt_pred, 'ModernBERT+AdaptFT': adaptft_pred, 'ModernBERT+Fusion': best_fusion_pred}
all_probs = {'TF-IDF+SVM': svm_prob, 'TF-IDF+LR': lr_prob, 'BERT-base': bert_prob, 'ModernBERT': mbert_prob, 'ModernBERT+TAPT': tapt_prob, 'ModernBERT+AdaptFT': adaptft_prob, 'ModernBERT+Fusion': best_fusion_prob}
for name in MODEL_ORDER:
    results[name].pop('TrainTime(s)', None)
    output = test_df[['row_id', 'text_hash', 'label']].copy()
    output['pred'] = all_preds[name]
    output['prob_positive'] = all_probs[name]
    output.to_csv(PRED_DIR / (name.replace('+', '_') + '_predictions.csv'), index=False)
    pd.DataFrame(confusion_matrix(test_df.label, all_preds[name], labels=[0, 1]), index=['Negative', 'Positive'], columns=['Negative', 'Positive']).to_csv(TABLE_DIR / (name.replace('+', '_') + '_confusion_matrix.csv'))
    pd.DataFrame(classification_report(test_df.label, all_preds[name], labels=[0, 1], target_names=['Negative', 'Positive'], output_dict=True, zero_division=0)).T.to_csv(TABLE_DIR / (name.replace('+', '_') + '_classification_report.csv'))
metrics = pd.DataFrame.from_dict(results, orient='index').reindex(MODEL_ORDER).rename_axis('Model').reset_index()
metrics.to_csv(TABLE_DIR / 'test_metrics.csv', index=False)
joblib.dump(svm_pipe, MODEL_DIR / 'tfidf_svm.joblib')
joblib.dump(lr_pipe, MODEL_DIR / 'tfidf_lr.joblib')
joblib.dump(final_pipe, MODEL_DIR / 'fusion_classifier.joblib')
(MODEL_DIR / 'fusion_config.json').write_text(json.dumps({'pooling': best_cfg['strategy'], 'C': best_cfg['C'], 'validation_macro_f1': best_val_f1, 'encoder': 'tapt_finetuned', 'max_length': 512, 'thumbs_available': THUMBS_AVAILABLE, 'thumbs_column': THUMBS_COL, 'developer_response_column': DEV_COL}, indent=2))
(RUN_DIR / 'run_config.json').write_text(json.dumps({'dataset': 'Dataset 1', 'seed': 42, 'input_sha256': hashlib.sha256(CSV_FILE.read_bytes()).hexdigest(), 'label_mapping': {'0': 'negative (1-2 stars)', '1': 'positive (4-5 stars)'}, 'max_length': 512, 'train_limit': 25000, 'validation_limit': 5000, 'test_limit': 5000, 'effective_training_batch': 32, 'tapt_epochs': 1, 'fine_tuning_epochs': 3, 'adaptft_epochs': 5, 'test_used_for_model_selection': False}, indent=2))
(RUN_DIR / 'environment.txt').write_text(subprocess.check_output([sys.executable, '-m', 'pip', 'freeze'], text=True))
display(metrics)
print("Results saved:", RUN_DIR)
